<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">از سطر و ستون تا ضرب ماتریسی</h1><p style="text-align:right"><b>پرسش آزمایش:</b> هر خانهٔ حاصل‌ضرب از کدام عددهای ورودی ساخته می‌شود؟</p><p style="text-align:right">پیش‌نیاز: <a target="_self" href="http://127.0.0.1:8000/part-02/chapter-02/06-dot.html"><bdi dir="ltr">06-dot</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-02/chapter-02/07-matmul.html"><bdi dir="ltr">07-matmul</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای اجرای دوباره از ابتدا، <bdi dir="ltr">Kernel</bdi> را <bdi dir="ltr">Restart</bdi> و سپس <bdi dir="ltr">Run All</bdi> کنید. برای بازکردن لینک درس‌ها، سرور کتاب باید روی پورت ۸۰۰۰ اجرا شده باشد؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">این مثالِ کوچک با فهرست‌های <bdi dir="ltr">Python</bdi> است، نه پیاده‌سازی دوم <bdi dir="ltr">Mini-GPT</bdi>. پیش از اجرا، شکل حاصل‌ضرب یک ماتریس دو‌در‌سه در یک ماتریس سه‌در‌دو را بنویسید و خانهٔ نخست را دستی حساب کنید.</p>
</div>

In [ ]:
def shape(a):
    if not a or not a[0] or any(len(row) != len(a[0]) for row in a):
        raise ValueError("Expected a nonempty rectangular matrix")
    return len(a), len(a[0])

def matmul(a, b):
    rows, inner = shape(a)
    b_rows, cols = shape(b)
    if inner != b_rows:
        raise ValueError(f"Inner dimensions differ: {inner} != {b_rows}")
    return [[sum(a[i][k] * b[k][j] for k in range(inner))
             for j in range(cols)] for i in range(rows)]

a = [[1, 2, 3], [4, 5, 6]]
b = [[1, 0], [0, 1], [1, 1]]
product = matmul(a, b)
print("A:", a, shape(a))
print("B:", b, shape(b))
print("AB:", product, shape(product))
print("First cell:", "1*1 + 2*0 + 3*1 =", product[0][0])
assert product == [[4, 5], [10, 11]]


<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">یک عدد را عوض کنید</h2><p style="text-align:right">اگر فقط خانهٔ سطر صفر و ستون یکِ <bdi dir="ltr">A</bdi> را یک واحد زیاد کنیم، کدام خانه‌های حاصل تغییر می‌کنند؟ پیش‌بینی را بنویسید؛ بعد نمودار اختلاف را ببینید.</p>
</div>

In [ ]:
import matplotlib.pyplot as plt
changed = [row[:] for row in a]
changed[0][1] += 1
new_product = matmul(changed, b)
difference = [[new_product[i][j] - product[i][j] for j in range(2)] for i in range(2)]
print("Change in AB:", difference)
assert difference == [[0, 1], [0, 0]]
fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for ax, values, title in zip(axes, [product, new_product, difference],
                             ["AB", "Changed AB", "Difference"]):
    ax.imshow(values, cmap="Blues")
    flat_values = [value for row in values for value in row]
    midpoint = (min(flat_values) + max(flat_values)) / 2
    for i, row in enumerate(values):
        for j, value in enumerate(row):
            ax.text(j, i, str(value), ha="center", va="center",
                    color="white" if value > midpoint else "black")
    ax.set(title=title, xlabel="Column", ylabel="Row",
           xticks=range(len(values[0])), yticks=range(len(values)))
plt.tight_layout()
plt.show()


<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خطا را بخوانید</h2><p style="text-align:right">ضرب <bdi dir="ltr">A</bdi> در <bdi dir="ltr">A</bdi> چرا تعریف نشده است؟ خطای عمدی را می‌گیریم تا اجرای دفتر متوقف نشود. سپس <bdi dir="ltr">A</bdi> در ترانهادهٔ <bdi dir="ltr">A</bdi> را حساب کنید. ضرب عضو‌به‌عضو جایگزین این عمل نیست.</p>
</div>

In [ ]:
try:
    matmul(a, a)
except ValueError as error:
    print("Expected failure:", error)
else:
    raise AssertionError("The dimension mismatch should fail")
transposed = [list(column) for column in zip(*a)]
print("A @ A.T:", matmul(a, transposed))
assert matmul(a, transposed) == [[14, 32], [32, 77]]


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>انتظار و برداشت:</b> خروجی دو‌در‌دو است. تغییر یک مؤلفهٔ سطر ورودی، سهم همان مؤلفه را در ستون‌های خروجی تغییر می‌دهد. توضیح دهید چرا در آزمایش ما فقط یک خانه تغییر کرد؛ این خاصیتِ عددهای <bdi dir="ltr">B</bdi> است، نه قانونی همیشگی.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی و نتیجهٔ اجرا را کنار هم بنویسید؛ اگر تفاوتی داشتند، دلیلش را توضیح دهید. سپس به <a target="_self" href="http://127.0.0.1:8000/part-02/chapter-02/07-matmul.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">تمرین تکمیلی: اثر یک تغییر را بدون تکرار کل ضرب حساب کنید</h2>
<p style="text-align:right">از معنای ضرب ماتریسی برای پیش‌بینی محل و مقدار تغییر خروجی استفاده کنید. پیش‌نیاز: نمونهٔ سطر و ستون در همین دفتر را اجرا کرده باشید. مثال‌های قبلی این دفتر را نگه داشته‌ایم. اکنون دو تابع <bdi dir="ltr">TODO</bdi> را خودتان بنویسید؛ <bdi dir="ltr">INCOMPLETE</bdi> یعنی کار هنوز تمام نشده است.</p>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر فقط <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">A[i,k]</code> به اندازهٔ <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">delta</code> زیاد شود، چه ارتباطی میان تغییر سطر <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">i</code> خروجی و سطر <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">k</code> ماتریس <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">B</code> وجود دارد؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
review_a = [[1.,2.,3.],[4.,5.,6.]]
review_b = [[1.,2.],[3.,4.],[5.,6.]]
print('A:',review_a,'B:',review_b)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">product_change(rows,b,i,k,delta)</code> فقط ماتریس اختلاف خروجی را با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">list</code> برگرداند. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">rows</code> تعداد سطرهای <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">A</code> است؛ بدون ساخت <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">A</code> یا تکرار <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">matmul</code>، سهم تغییر یک خانه را حساب کنید.</p>
</div>

In [ ]:
def product_change(rows, b, i, k, delta):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = product_change(2,review_b,0,1,2.)
    if result is None: return False
    assert result == [[6.,8.],[0.,0.]]
    assert product_change(3,[[2.,-1.,4.]],2,0,-0.5) == [[0.,0.,0.],[0.,0.,0.],[-1.,0.5,-2.]]
    assert product_change(1,review_b,0,2,0.) == [[0.,0.]]
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">delta</code> را از ۱ به ۲ و منفی ۱ تغییر دهید. محل اثر ثابت است؛ مقدار و علامت آن چه می‌شود؟</p>
</div>

In [ ]:
for delta in (1.,2.,-1.):
    print('delta and changed output row:',delta,[delta*value for value in review_b[1]])

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">[[0]*cols]*rows</code> همهٔ سطرها یک شیء می‌شوند. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">zero_matrix(rows,cols)</code> را اصلاح کنید تا دست‌کاری یک سطر به دیگری سرایت نکند.</p>
</div>

In [ ]:
wrong = [[0.]*2]*3
wrong[0][0] = 9.
print('aliased rows:',wrong)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def zero_matrix(rows, cols):
    # TODO
    return None

In [ ]:
def test_repair():
    result = zero_matrix(3,2)
    if result is None: return False
    assert result == [[0.,0.],[0.,0.],[0.,0.]]
    result[0][0] = 7.
    assert result[1][0] == result[2][0] == 0.
    assert zero_matrix(1,3) == [[0.,0.,0.]]
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right">در <bdi dir="ltr">Projection</bdi>های <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">MiniGPT</code> نیز هر مؤلفهٔ ورودی از راه یک ردیف یا ستون وزن به چند خروجی سهم می‌دهد. این تمرین مسیر اثر را پیش از ورود به <bdi dir="ltr">Tensor</bdi>ها قابل محاسبه می‌کند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">چرا تغییر یک خانهٔ <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">A</code> گاهی چند خانهٔ خروجی را عوض می‌کند، با اینکه در مثال اولیهٔ دفتر فقط یک خانه تغییر کرد؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-02/chapter-02/07-matmul.html">بازگشت به درس مرتبط</a> · <a target="_self" href="http://127.0.0.1:8000/answers/lab-01_matrix_products.html">فقط پس از تلاش: پاسخ مرجع تمرین تکمیلی</a></p></div>